# 3.12 — Quantile & Isotonic Regression

Quantile regression changes the target from the conditional mean to a chosen conditional percentile, while isotonic regression adds a shape constraint: predictions must move monotonically with a sorted feature. In this lesson, you will build the pinball loss, percentile fits, monotone pooling, validation scoring, and stability checks from scratch with NumPy so the optimization target is always visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build quantile and isotonic regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so nothing is a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, losses, sorting, and small optimization grids.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy data and jittered examples.

### 1. Pinball loss makes under- and over-prediction cost different amounts

Quantile regression begins by changing the loss. For residual $r=y-\hat y$, the pinball loss is $\rho_\tau(r)=r(\tau-\mathbf 1[r<0])$. If $r>0$, the model predicted too low and pays $\tau r$; if $r<0$, it predicted too high and pays $(1-\tau)(-r)$. At $\tau=0.8$, missing below the truth is four times as costly as overshooting by the same amount.

In [ ]:
def pinball_w(y_w, pred_w, tau_w):  # compute quantile loss for arrays or scalars.
    r_w = np.asarray(y_w, dtype=float) - np.asarray(pred_w, dtype=float)  # residual = truth - prediction.
    return np.where(r_w >= 0, tau_w * r_w, (tau_w - 1) * r_w)  # two linear pieces meeting at zero.

residuals_w = np.array([-2., -1., 0., 1., 2.])  # negative = over-prediction; positive = under-prediction.
losses_w = np.where(residuals_w >= 0, 0.8 * residuals_w, (0.8 - 1) * residuals_w)  # tau = 0.8.
print("residuals:", residuals_w)  # inspect signed mistakes.
print("losses at tau=.8:", np.round(losses_w, 2))  # inspect asymmetric penalties.
assert np.allclose(losses_w, [0.4, 0.2, 0.0, 0.8, 1.6])  # concrete pinball-loss check.

▶ What you'll see: positive residuals cost more than equal-size negative residuals when `tau=0.8`.

In [ ]:
rgrid_w = np.linspace(-3, 3, 121)  # possible residuals to visualize.
loss20_w = np.where(rgrid_w >= 0, 0.2 * rgrid_w, (0.2 - 1) * rgrid_w)  # low quantile loss.
loss80_w = np.where(rgrid_w >= 0, 0.8 * rgrid_w, (0.8 - 1) * rgrid_w)  # high quantile loss.
plt.figure(figsize=(5, 3))  # compact loss curve.
plt.plot(rgrid_w, loss20_w, label="tau=.2")  # lower quantile tilts left.
plt.plot(rgrid_w, loss80_w, label="tau=.8")  # upper quantile tilts right.
plt.axvline(0, color="black", linewidth=0.8)  # the kink at perfect prediction.
plt.xlabel("residual y - prediction")  # residual sign matters.
plt.ylabel("loss")  # loss height.
plt.title("1: pinball loss tilts around zero")  # name the concept.
plt.legend()  # identify curves.
plt.show()  # display plot.

▶ What you'll see: the steeper side swaps as `tau` changes, so the model is pushed toward a different percentile.

*Why it's done this way:* squared loss has equal slopes on both sides of zero, so it balances errors at the mean. Pinball loss deliberately changes the slopes; the minimizer must leave about a fraction `tau` of observations below it, which is the definition of a quantile.

### 2. A constant quantile minimizes average pinball risk

Before fitting a line, fit one number. Empirical risk is the average loss over the sample, and with pinball loss the best constant is an empirical quantile. The minimum occurs where moving the prediction up would create too many over-predictions, while moving it down would create too many expensive under-predictions.

In [ ]:
y_w = np.array([1., 2., 2., 4., 7.])  # small response sample with one high value.
tau_w = 0.8  # target an upper percentile.
candidates_w = np.linspace(1, 7, 121)  # candidate constants.
risks_w = np.array([pinball_w(y_w, q_w, tau_w).mean() for q_w in candidates_w])  # average pinball risk.
best_q_w = float(candidates_w[np.where(np.isclose(risks_w, risks_w.min(), atol=1e-12))[0][0]])  # first grid minimizer.
mean_w = float(y_w.mean())  # squared-loss target.
print("mean:", round(mean_w, 2), "best pinball constant:", round(best_q_w, 2))  # compare targets.
assert round(mean_w, 2) == 3.2 and abs(best_q_w - 4.0) < 0.06  # concrete checks.

▶ What you'll see: the upper quantile is near 4, above the mean 3.2.

In [ ]:
q80_w = float(np.quantile(y_w, 0.8, method="lower"))  # one empirical convention for the 80th percentile.
print("risk at mean:", round(pinball_w(y_w, mean_w, tau_w).mean(), 3))  # asymmetric risk at mean.
print("risk at q80:", round(pinball_w(y_w, q80_w, tau_w).mean(), 3))  # asymmetric risk at quantile.
plt.figure(figsize=(5, 3))  # risk curve.
plt.plot(candidates_w, risks_w, color="purple")  # ERM objective.
plt.axvline(mean_w, color="gray", linestyle="--", label="mean")  # mean reference.
plt.axvline(q80_w, color="seagreen", linestyle="--", label="80% quantile")  # quantile reference.
plt.xlabel("constant prediction")  # candidate q.
plt.ylabel("mean pinball loss")  # risk.
plt.title("2: ERM chooses a quantile")  # concept title.
plt.legend()  # labels.
plt.show()  # display curve.

▶ What you'll see: the risk curve bottoms out near the empirical upper quantile, not at the mean.

*Why it's done this way:* ERM only knows the loss you hand it. With pinball loss, raising a too-low prediction saves expensive right-tail losses until enough points are below the prediction; the slope balance is exactly the quantile condition.

### 3. Linear quantile regression is ERM over lines

A conditional quantile line uses predictions $\hat y=b_0+b_1x$. Large libraries use efficient convex optimization, but a tiny grid search exposes the same objective: evaluate average pinball loss for many slopes and intercepts, then keep the best one.

In [ ]:
x_w = np.array([0., 1., 2., 3., 4., 5.])  # one feature.
y_w = np.array([1.0, 1.7, 2.4, 3.8, 5.8, 7.5])  # response with increasing spread.
tau_w = 0.8  # fit an upper conditional quantile.
slopes_w = np.linspace(0.8, 1.5, 71)  # candidate b1 values.
intercepts_w = np.linspace(0.5, 1.8, 66)  # candidate b0 values.
print("grid size:", len(slopes_w) * len(intercepts_w))  # inspect the search budget.

▶ What you'll see: a small enough grid to understand, large enough to find a sensible line.

In [ ]:
best_loss_w = np.inf  # best empirical risk seen so far.
best_pair_w = None  # best (intercept, slope).
for b1_w in slopes_w:  # try slopes.
    for b0_w in intercepts_w:  # try intercepts.
        pred_w = b0_w + b1_w * x_w  # candidate quantile line.
        loss_w = float(pinball_w(y_w, pred_w, tau_w).mean())  # average pinball loss.
        if loss_w < best_loss_w:  # keep the empirical-risk minimizer.
            best_loss_w = loss_w  # update risk.
            best_pair_w = (b0_w, b1_w)  # update parameters.
print("best intercept, slope:", np.round(best_pair_w, 3), "loss:", round(best_loss_w, 3))  # inspect fit.
assert round(best_loss_w, 3) <= 0.155  # concrete quality check.

▶ What you'll see: the selected line is high enough to make under-prediction relatively rare.

In [ ]:
b0_w, b1_w = best_pair_w  # unpack fitted line.
pred_w = b0_w + b1_w * x_w  # fitted predictions.
coverage_w = float(np.mean(y_w <= pred_w))  # empirical fraction below the line.
print("coverage:", round(coverage_w, 3))  # tiny-sample calibration check.
assert coverage_w >= 0.66  # exact .8 is impossible to demand from six points.
plt.figure(figsize=(5, 3))  # fit plot.
plt.scatter(x_w, y_w, color="black", label="data")  # observed points.
plt.plot(x_w, pred_w, color="crimson", label="tau=.8 fit")  # fitted upper quantile.
plt.xlabel("x")  # feature.
plt.ylabel("y")  # response.
plt.title("3: quantile line by grid-search ERM")  # concept title.
plt.legend()  # labels.
plt.show()  # display fit.

▶ What you'll see: the quantile line rides above most points because missing high is expensive at `tau=0.8`.

*Why it's done this way:* the model family provides candidate predictions, pinball loss defines the percentile error, and averaging creates the empirical risk. Grid search is slow but transparent; the mathematical target is the same as in production solvers.

### 4. Isotonic regression pools adjacent violators

Isotonic regression fits predictions that must be nondecreasing with sorted $x$. The core algorithm is pooling adjacent violators: start with one block per point, merge neighboring blocks whose means are out of order, and assign the merged weighted average to the whole block.

In [ ]:
x_iso_w = np.arange(8)  # sorted feature positions.
y_iso_w = np.array([1.0, 2.2, 1.7, 3.0, 2.6, 4.2, 4.0, 5.1])  # noisy increasing pattern.
print("raw y:", y_iso_w)  # inspect observations.
print("adjacent differences:", np.round(np.diff(y_iso_w), 2))  # negative entries violate monotonicity.
assert np.any(np.diff(y_iso_w) < 0)  # data contains real violations.

▶ What you'll see: several adjacent drops break the nondecreasing rule.

In [ ]:
levels_w = list(y_iso_w.astype(float))  # current block means.
weights_w = [1] * len(levels_w)  # current block sizes.
starts_w = list(range(len(y_iso_w)))  # block starts.
ends_w = list(range(len(y_iso_w)))  # block ends.
i_w = 0  # scan pointer.
while i_w < len(levels_w) - 1:  # scan adjacent block means.
    if levels_w[i_w] > levels_w[i_w + 1]:  # violation found.
        total_w = weights_w[i_w] + weights_w[i_w + 1]  # merged block size.
        avg_w = (levels_w[i_w] * weights_w[i_w] + levels_w[i_w + 1] * weights_w[i_w + 1]) / total_w  # weighted average.
        levels_w[i_w:i_w + 2] = [avg_w]  # replace two blocks by one block.
        weights_w[i_w:i_w + 2] = [total_w]  # replace two weights by combined weight.
        ends_w[i_w:i_w + 2] = [ends_w[i_w + 1]]  # merged block reaches the right endpoint.
        starts_w[i_w:i_w + 2] = [starts_w[i_w]]  # merged block starts at the left endpoint.
        i_w = max(i_w - 1, 0)  # step back because a merge can create a new left violation.
    else:
        i_w += 1  # ordered pair, move right.
print("block levels:", np.round(levels_w, 3), "weights:", weights_w)  # inspect pooled blocks.
assert np.all(np.diff(levels_w) >= -1e-12)  # block means are monotone.

▶ What you'll see: violating neighbors become larger flat blocks with ordered means.

In [ ]:
fit_iso_w = np.empty_like(y_iso_w, dtype=float)  # allocate fitted values.
for level_w, start_w, end_w in zip(levels_w, starts_w, ends_w):  # expand each block.
    fit_iso_w[start_w:end_w + 1] = level_w  # every point in a block gets its block average.
print("isotonic fit:", np.round(fit_iso_w, 3))  # inspect final staircase.
assert np.all(np.diff(fit_iso_w) >= -1e-12)  # fitted sequence is nondecreasing.
plt.figure(figsize=(5, 3))  # raw vs fit plot.
plt.plot(x_iso_w, y_iso_w, "o--", label="raw y")  # noisy observations.
plt.step(x_iso_w, fit_iso_w, where="mid", color="seagreen", label="isotonic fit")  # monotone staircase.
plt.xlabel("sorted x")  # monotone direction.
plt.ylabel("prediction")  # fitted value.
plt.title("4: PAVA creates a monotone staircase")  # concept title.
plt.legend()  # labels.
plt.show()  # display plot.

▶ What you'll see: the fitted curve flattens wherever raw data tried to decrease.

*Why it's done this way:* if two adjacent block means violate the order constraint, no least-squares optimum can keep them separate. The weighted average is the closest shared value for the merged block, so repeated pooling gives the monotone projection.

### 5. Full decision scores include raw fit, cost, gap, and stabilization

The lesson's selection arithmetic follows the empirical-risk contract. Compute a raw average loss, add the method's cost, compare a tempting flexible alternative, and ask whether a stabilizing constraint changes the decision.

In [ ]:
losses_score_w = np.array([0.213, 0.083, 0.437])  # verified toy per-example losses.
raw_score_w = round(float(losses_score_w.mean()), 3)  # empirical risk R_S, rounded like the lesson arithmetic.
cost_w = 0.050  # complexity, regularization, or operational cost.
decision_score_w = raw_score_w + cost_w  # full score used for selection.
print("raw average:", round(raw_score_w, 3))  # inspect R_S.
print("decision score:", round(decision_score_w, 3))  # inspect R_S + cost.
assert round(raw_score_w, 3) == 0.244 and round(decision_score_w, 3) == 0.294  # lesson numbers.

▶ What you'll see: the raw average is 0.244, but the selectable score is 0.294 after cost.

In [ ]:
alternative_w = 0.338  # more flexible alternative's full score.
gap_w = alternative_w - decision_score_w  # absolute evidence gap.
relative_gap_w = gap_w / alternative_w  # scale-aware evidence gap.
stable_w = 0.80 * decision_score_w  # stabilizing knob reduces the score by 20%.
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))  # inspect comparison.
print("stabilized:", round(stable_w, 3))  # inspect stability score.
assert round(gap_w, 3) == 0.044 and round(relative_gap_w, 3) == 0.130 and round(stable_w, 3) == 0.235  # checks.

▶ What you'll see: the flexible option is worse by 0.044, and stabilization lowers the toy score to 0.235.

In [ ]:
labels_w = ["baseline", "flexible", "stabilized"]  # comparable options.
scores_w = np.array([decision_score_w, alternative_w, stable_w])  # full decision scores.
winner_w = labels_w[int(np.argmin(scores_w))]  # lower score wins.
print("winner:", winner_w, "score:", round(float(scores_w.min()), 3))  # final selection.
plt.figure(figsize=(5, 3))  # decision chart.
plt.bar(labels_w, scores_w, color=["steelblue", "orange", "seagreen"])  # compare full scores.
plt.ylabel("decision score (lower is better)")  # score axis.
plt.title("5: compare complete scores")  # concept title.
plt.show()  # display chart.

▶ What you'll see: the stabilized version has the lowest complete score in this verified toy case.

*Why it's done this way:* raw fit alone can reward brittle flexibility. The cost term, validation gap, and stability adjustment keep selection aligned with future performance rather than training convenience.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, losses, grids, sorting, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for loss curves, fitted lines, and monotone step plots.
np.random.seed(0)  # make every randomized example reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Compute pinball loss values

**Goal.** Evaluate asymmetric loss on simple residuals, because the sign of the error determines how a quantile model is punished. We build it in 2 steps.

In [ ]:
residuals_b1 = np.array([-2., -1., 0., 1., 2.])  # residual y - y_hat.
tau_b1 = 0.8  # upper quantile target.
print("residuals:", residuals_b1)  # inspect signed errors.

▶ What you'll see: equal-size residuals appear on both sides of zero.

In [ ]:
loss_b1 = np.where(residuals_b1 >= 0, tau_b1 * residuals_b1, (tau_b1 - 1) * residuals_b1)  # pinball formula.
print("losses:", np.round(loss_b1, 2))  # inspect asymmetric penalties.
assert np.allclose(loss_b1, [0.4, 0.2, 0.0, 0.8, 1.6])  # verify exact values.
plt.figure(figsize=(4, 3))  # compact bar chart.
plt.bar([str(r_b1) for r_b1 in residuals_b1], loss_b1, color="purple")  # residual-to-loss map.
plt.title("Basic 1: pinball loss")  # title.
plt.xlabel("residual")  # x-axis.
plt.ylabel("loss")  # y-axis.
plt.show()  # display.

▶ What you'll see: under-predictions are much taller bars at `tau=0.8`.

👀 Takeaway: quantile regression begins by tilting the residual penalty.

### Basic 2 — Compare the mean to an upper quantile

**Goal.** Compute a mean and an empirical quantile from the same data, because they answer different questions. We build it in 2 steps.

In [ ]:
y_b2 = np.array([1., 2., 2., 4., 7.])  # small sample.
mean_b2 = float(np.mean(y_b2))  # average target.
q80_b2 = float(np.quantile(y_b2, 0.8, method="lower"))  # empirical upper quantile.
print("mean:", round(mean_b2, 2), "q80:", q80_b2)  # compare summaries.
assert round(mean_b2, 2) == 3.2 and q80_b2 == 4.0  # concrete checks.

▶ What you'll see: the 80th percentile is higher than the mean.

In [ ]:
plt.figure(figsize=(4, 3))  # one-dimensional value plot.
plt.scatter(y_b2, np.zeros_like(y_b2), color="black")  # sample values.
plt.axvline(mean_b2, color="gray", linestyle="--", label="mean")  # mean marker.
plt.axvline(q80_b2, color="seagreen", linestyle="--", label="q80")  # quantile marker.
plt.yticks([])  # hide unused axis.
plt.title("Basic 2: mean vs quantile")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the quantile marker sits farther right to represent the upper tail.

👀 Takeaway: a quantile is not a noisy mean; it is a different target.

### Basic 3 — Average pinball losses

**Goal.** Compute empirical risk for one candidate prediction, because ERM averages one loss per example. We build it in 2 steps.

In [ ]:
y_b3 = np.array([1., 2., 2., 4., 7.])  # observations.
pred_b3 = 4.0  # candidate constant.
tau_b3 = 0.8  # upper quantile.
resid_b3 = y_b3 - pred_b3  # residuals.
print("residuals:", resid_b3)  # inspect signs.

▶ What you'll see: the large point at 7 is under-predicted.

In [ ]:
losses_b3 = np.where(resid_b3 >= 0, tau_b3 * resid_b3, (tau_b3 - 1) * resid_b3)  # pinball losses.
risk_b3 = float(losses_b3.mean())  # average loss.
print("losses:", np.round(losses_b3, 2), "risk:", round(risk_b3, 3))  # inspect contributions and average.
assert round(risk_b3, 3) == 0.76  # verify hand arithmetic.
plt.figure(figsize=(4, 3))  # loss contribution plot.
plt.bar(range(len(y_b3)), losses_b3, color="steelblue")  # per-example losses.
plt.title("Basic 3: empirical pinball risk")  # title.
plt.xlabel("example")  # x-axis.
plt.ylabel("loss")  # y-axis.
plt.show()  # display.

▶ What you'll see: the high observation dominates the loss because it is expensive to miss low.

👀 Takeaway: quantile ERM is still ordinary average-loss minimization.

### Basic 4 — Search for the best constant quantile

**Goal.** Minimize average pinball loss over constants, because the minimizer should be the empirical quantile. We build it in 3 steps.

In [ ]:
y_b4 = np.array([1., 2., 2., 4., 7.])  # data.
tau_b4 = 0.8  # target quantile.
grid_b4 = np.linspace(1, 7, 121)  # candidate constants.
print("candidates:", len(grid_b4))  # inspect grid resolution.

▶ What you'll see: the candidate predictions cover the data range.

In [ ]:
risks_b4 = []  # store average losses.
for q_b4 in grid_b4:  # evaluate each candidate.
    r_b4 = y_b4 - q_b4  # residuals.
    losses_b4 = np.where(r_b4 >= 0, tau_b4 * r_b4, (tau_b4 - 1) * r_b4)  # pinball losses.
    risks_b4.append(float(losses_b4.mean()))  # average risk.
risks_b4 = np.array(risks_b4)  # array for argmin.
best_b4 = float(grid_b4[np.where(np.isclose(risks_b4, risks_b4.min(), atol=1e-12))[0][0]])  # first best constant.
print("best constant:", round(best_b4, 2))  # inspect fit.
assert abs(best_b4 - 4.0) < 0.06  # verify quantile target.

In [ ]:
plt.figure(figsize=(4, 3))  # risk curve.
plt.plot(grid_b4, risks_b4, color="purple")  # objective.
plt.axvline(best_b4, color="seagreen", linestyle="--")  # minimizer.
plt.title("Basic 4: constant quantile ERM")  # title.
plt.xlabel("q")  # candidate.
plt.ylabel("mean loss")  # risk.
plt.show()  # display.

▶ What you'll see: the curve reaches its minimum near 4.

👀 Takeaway: the pinball-risk minimizer is an empirical percentile.

### Basic 5 — Plot two pinball losses

**Goal.** Visualize low and high quantile losses, because `tau` controls which side is steep. We build it in 2 steps.

In [ ]:
resid_b5 = np.linspace(-3, 3, 121)  # residual grid.
loss20_b5 = np.where(resid_b5 >= 0, 0.2 * resid_b5, (0.2 - 1) * resid_b5)  # tau .2.
loss80_b5 = np.where(resid_b5 >= 0, 0.8 * resid_b5, (0.8 - 1) * resid_b5)  # tau .8.
print("loss at residual +2:", round(float(loss20_b5[100]), 2), round(float(loss80_b5[100]), 2))  # compare slopes.

▶ What you'll see: the same positive residual costs more under the upper quantile loss.

In [ ]:
plt.figure(figsize=(4, 3))  # curve figure.
plt.plot(resid_b5, loss20_b5, label="tau=.2")  # lower quantile.
plt.plot(resid_b5, loss80_b5, label="tau=.8")  # upper quantile.
plt.axvline(0, color="black", linewidth=0.8)  # kink.
plt.title("Basic 5: loss tilt")  # title.
plt.xlabel("residual")  # x-axis.
plt.ylabel("loss")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the curves tilt in opposite directions around zero.

👀 Takeaway: changing `tau` changes the business cost encoded by the model.

### Basic 6 — Fit a quantile line by grid search

**Goal.** Search line parameters for an upper quantile fit, because quantile regression is ERM over predictions. We build it in 3 steps.

In [ ]:
x_b6 = np.array([0., 1., 2., 3., 4., 5.])  # feature.
y_b6 = np.array([1.0, 1.7, 2.4, 3.8, 5.8, 7.5])  # response.
slopes_b6 = np.linspace(0.8, 1.5, 36)  # candidate slopes.
intercepts_b6 = np.linspace(0.5, 1.8, 27)  # candidate intercepts.
print("grid size:", len(slopes_b6) * len(intercepts_b6))  # inspect search size.

▶ What you'll see: a finite set of candidate lines.

In [ ]:
best_loss_b6 = np.inf  # initialize best loss.
best_b6 = None  # initialize best coefficients.
for b1_b6 in slopes_b6:  # search slopes.
    for b0_b6 in intercepts_b6:  # search intercepts.
        pred_b6 = b0_b6 + b1_b6 * x_b6  # line predictions.
        r_b6 = y_b6 - pred_b6  # residuals.
        loss_b6 = np.where(r_b6 >= 0, 0.8 * r_b6, (0.8 - 1) * r_b6).mean()  # mean pinball loss.
        if loss_b6 < best_loss_b6:  # update best.
            best_loss_b6 = float(loss_b6)  # save risk.
            best_b6 = (float(b0_b6), float(b1_b6))  # save line.
print("best b0,b1:", np.round(best_b6, 3), "loss:", round(best_loss_b6, 3))  # inspect fit.
assert round(best_loss_b6, 3) <= 0.17  # quality check.

In [ ]:
pred_line_b6 = best_b6[0] + best_b6[1] * x_b6  # fitted predictions.
plt.figure(figsize=(4, 3))  # fit plot.
plt.scatter(x_b6, y_b6, color="black")  # data.
plt.plot(x_b6, pred_line_b6, color="crimson")  # quantile line.
plt.title("Basic 6: tau=.8 line")  # title.
plt.xlabel("x")  # x-axis.
plt.ylabel("y")  # y-axis.
plt.show()  # display.

▶ What you'll see: the selected line sits high to avoid expensive under-prediction.

👀 Takeaway: quantile regression changes the line by changing the loss, not by changing the data.

### Basic 7 — Check quantile coverage

**Goal.** Count how many observations fall below a fitted quantile line, because quantiles should be checked by coverage. We build it in 2 steps.

In [ ]:
x_b7 = x_b6.copy()  # reuse same section's feature values.
y_b7 = y_b6.copy()  # reuse responses.
pred_b7 = pred_line_b6.copy()  # reuse fitted line.
below_b7 = y_b7 <= pred_b7  # coverage indicator.
print("below indicators:", below_b7.astype(int))  # inspect covered points.

▶ What you'll see: most points lie below the upper quantile line.

In [ ]:
coverage_b7 = float(np.mean(below_b7))  # fraction below line.
print("coverage:", round(coverage_b7, 3))  # inspect empirical coverage.
assert coverage_b7 >= 0.66  # tiny-sample tolerance.
plt.figure(figsize=(4, 3))  # coverage plot.
plt.scatter(x_b7[below_b7], y_b7[below_b7], color="seagreen", label="below")  # covered points.
plt.scatter(x_b7[~below_b7], y_b7[~below_b7], color="crimson", label="above")  # uncovered points.
plt.plot(x_b7, pred_b7, color="black")  # fitted quantile.
plt.title("Basic 7: coverage")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: a few points may remain above the line, especially in a tiny sample.

👀 Takeaway: a quantile model is judged partly by whether its coverage matches the requested percentile.

### Basic 8 — Detect monotone violations

**Goal.** Identify adjacent drops in sorted predictions, because isotonic regression repairs these violations. We build it in 2 steps.

In [ ]:
y_b8 = np.array([1.0, 2.2, 1.7, 3.0, 2.6, 4.2, 4.0, 5.1])  # noisy sorted-by-x values.
diff_b8 = np.diff(y_b8)  # adjacent changes.
violations_b8 = np.where(diff_b8 < 0)[0]  # starts of decreasing pairs.
print("differences:", np.round(diff_b8, 2))  # inspect monotone changes.
print("violation starts:", violations_b8)  # inspect drops.
assert np.array_equal(violations_b8, np.array([1, 3, 5]))  # known violations.

▶ What you'll see: negative differences at three locations.

In [ ]:
plt.figure(figsize=(4, 3))  # raw sequence plot.
plt.plot(np.arange(len(y_b8)), y_b8, "o--", color="gray")  # noisy values.
plt.scatter(violations_b8 + 1, y_b8[violations_b8 + 1], color="red", label="drop")  # lower endpoint of each drop.
plt.title("Basic 8: monotone violations")  # title.
plt.xlabel("sorted index")  # x-axis.
plt.ylabel("value")  # y-axis.
plt.legend()  # label drops.
plt.show()  # display.

▶ What you'll see: red points mark where the sequence falls instead of rising.

👀 Takeaway: isotonic regression is useful only after you know which order should be respected.

### Basic 9 — Pool one violation

**Goal.** Average an out-of-order pair, because pooling creates the closest flat value satisfying local monotonicity. We build it in 2 steps.

In [ ]:
left_b9 = 2.2  # first value.
right_b9 = 1.7  # next value is lower.
pooled_b9 = (left_b9 + right_b9) / 2  # equal-weight pooled level.
print("pooled level:", pooled_b9)  # inspect local repair.
assert round(pooled_b9, 2) == 1.95  # arithmetic check.

▶ What you'll see: the violating pair is replaced by a shared value between them.

In [ ]:
plt.figure(figsize=(4, 3))  # local repair plot.
plt.plot([0, 1], [left_b9, right_b9], "o--", label="raw")  # decreasing pair.
plt.plot([0, 1], [pooled_b9, pooled_b9], "s-", label="pooled")  # flat monotone repair.
plt.xticks([0, 1], ["left", "right"])  # pair labels.
plt.ylabel("value")  # y-axis.
plt.title("Basic 9: pooled violator")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the decreasing segment becomes flat.

👀 Takeaway: PAVA is repeated local averaging of adjacent violations.

### Basic 10 — Add cost to empirical risk

**Goal.** Reproduce the lesson's raw and decision scores, because model selection should compare the full score. We build it in 2 steps.

In [ ]:
losses_b10 = np.array([0.213, 0.083, 0.437])  # verified per-example losses.
raw_b10 = float(losses_b10.mean())  # empirical risk.
cost_b10 = 0.050  # complexity or operational cost.
score_b10 = raw_b10 + cost_b10  # full score.
print("raw:", round(raw_b10, 3), "score:", round(score_b10, 3))  # inspect arithmetic.
assert round(raw_b10, 3) == 0.244 and round(score_b10, 3) == 0.294  # lesson checks.

▶ What you'll see: the full score is larger than the raw fit.

In [ ]:
plt.figure(figsize=(4, 3))  # composition chart.
plt.bar(["raw", "cost", "total"], [raw_b10, cost_b10, score_b10], color=["steelblue", "orange", "seagreen"])  # pieces.
plt.title("Basic 10: decision score")  # title.
plt.ylabel("score")  # y-axis.
plt.show()  # display.

▶ What you'll see: the cost term visibly changes the selection quantity.

👀 Takeaway: optimizing raw training loss alone can choose the wrong model.

## 🟡 Easy

### Easy 1 — Fit lower, median, and upper constants

**Goal.** Fit multiple constant quantiles, because intervals require more than one percentile. We build it in 3 steps.

In [ ]:
y_e1 = np.array([1., 2., 2., 4., 7., 9.])  # skewed sample.
taus_e1 = np.array([0.2, 0.5, 0.8])  # lower, middle, upper.
grid_e1 = np.linspace(1, 9, 161)  # candidate constants.
print("taus:", taus_e1)  # inspect requested levels.

▶ What you'll see: three quantile targets will be fit independently.

In [ ]:
best_q_e1 = []  # fitted constants.
for tau_e1 in taus_e1:  # loop over quantile levels.
    risks_e1 = []  # risks for this tau.
    for q_e1 in grid_e1:  # candidate constant.
        r_e1 = y_e1 - q_e1  # residuals.
        risks_e1.append(float(np.where(r_e1 >= 0, tau_e1 * r_e1, (tau_e1 - 1) * r_e1).mean()))  # average loss.
    best_q_e1.append(float(grid_e1[np.argmin(risks_e1)]))  # minimizer.
best_q_e1 = np.array(best_q_e1)  # array for checks and plot.
print("fitted constants:", np.round(best_q_e1, 2))  # inspect order.
assert np.all(np.diff(best_q_e1) >= 0)  # higher quantiles should not be lower in this example.

In [ ]:
plt.figure(figsize=(4, 3))  # interval summary.
plt.scatter(y_e1, np.zeros_like(y_e1), color="black", label="data")  # sample points.
for tau_e1, q_e1 in zip(taus_e1, best_q_e1):  # draw each fitted quantile.
    plt.axvline(q_e1, linestyle="--", label=f"tau={tau_e1}")  # quantile line.
plt.yticks([])  # remove y-axis ticks.
plt.title("Easy 1: constant quantile band")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: fitted constants move right as `tau` increases.

👀 Takeaway: quantile levels can be stacked to describe uncertainty, not just a single prediction.

### Easy 2 — Compare mean and quantile regression lines

**Goal.** Fit a least-squares mean line and an upper-quantile line, because different losses answer different prediction questions. We build it in 3 steps.

In [ ]:
x_e2 = np.array([0., 1., 2., 3., 4., 5.])  # feature.
y_e2 = np.array([1.0, 1.7, 2.4, 3.8, 5.8, 7.5])  # response.
X_e2 = np.vstack([np.ones_like(x_e2), x_e2]).T  # design matrix.
coef_mean_e2 = np.linalg.solve(X_e2.T @ X_e2, X_e2.T @ y_e2)  # least-squares coefficients.
print("mean-line coefficients:", np.round(coef_mean_e2, 3))  # inspect mean fit.

▶ What you'll see: the least-squares line targets the conditional average.

In [ ]:
best_loss_e2 = np.inf  # best pinball risk.
best_coef_e2 = None  # best quantile coefficients.
for b1_e2 in np.linspace(0.8, 1.5, 36):  # candidate slopes.
    for b0_e2 in np.linspace(0.5, 1.8, 27):  # candidate intercepts.
        pred_e2 = b0_e2 + b1_e2 * x_e2  # candidate line.
        r_e2 = y_e2 - pred_e2  # residuals.
        loss_e2 = np.where(r_e2 >= 0, 0.8 * r_e2, (0.8 - 1) * r_e2).mean()  # upper-quantile risk.
        if loss_e2 < best_loss_e2:  # update best.
            best_loss_e2 = float(loss_e2)  # save risk.
            best_coef_e2 = np.array([b0_e2, b1_e2])  # save coefficients.
print("quantile coefficients:", np.round(best_coef_e2, 3))  # inspect upper fit.
assert best_loss_e2 <= 0.17  # quality check.

In [ ]:
plt.figure(figsize=(4, 3))  # comparison plot.
plt.scatter(x_e2, y_e2, color="black")  # data.
plt.plot(x_e2, X_e2 @ coef_mean_e2, color="gray", label="mean")  # least-squares fit.
plt.plot(x_e2, best_coef_e2[0] + best_coef_e2[1] * x_e2, color="crimson", label="tau=.8")  # quantile fit.
plt.title("Easy 2: mean vs quantile line")  # title.
plt.xlabel("x")  # x-axis.
plt.ylabel("y")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the quantile line tends to ride above the mean line.

👀 Takeaway: choosing pinball loss changes the fitted function's meaning.

### Easy 3 — Implement full PAVA

**Goal.** Run pooling adjacent violators end to end, because it is the core isotonic-regression algorithm. We build it in 3 steps.

In [ ]:
y_e3 = np.array([1.0, 2.2, 1.7, 3.0, 2.6, 4.2, 4.0, 5.1])  # noisy sequence.
levels_e3 = list(y_e3.astype(float))  # block means.
weights_e3 = [1] * len(y_e3)  # block sizes.
starts_e3 = list(range(len(y_e3)))  # block starts.
ends_e3 = list(range(len(y_e3)))  # block ends.
print("raw sequence:", y_e3)  # inspect input.

▶ What you'll see: the raw values violate monotonicity.

In [ ]:
i_e3 = 0  # scan pointer.
while i_e3 < len(levels_e3) - 1:  # scan adjacent blocks.
    if levels_e3[i_e3] > levels_e3[i_e3 + 1]:  # violation.
        total_e3 = weights_e3[i_e3] + weights_e3[i_e3 + 1]  # combined size.
        avg_e3 = (levels_e3[i_e3] * weights_e3[i_e3] + levels_e3[i_e3 + 1] * weights_e3[i_e3 + 1]) / total_e3  # pooled mean.
        levels_e3[i_e3:i_e3 + 2] = [avg_e3]  # merge levels.
        weights_e3[i_e3:i_e3 + 2] = [total_e3]  # merge weights.
        ends_e3[i_e3:i_e3 + 2] = [ends_e3[i_e3 + 1]]  # merge end.
        starts_e3[i_e3:i_e3 + 2] = [starts_e3[i_e3]]  # merge start.
        i_e3 = max(i_e3 - 1, 0)  # recheck left.
    else:
        i_e3 += 1  # move right.
print("levels:", np.round(levels_e3, 3), "weights:", weights_e3)  # inspect blocks.
assert np.all(np.diff(levels_e3) >= -1e-12)  # monotone block means.

In [ ]:
fit_e3 = np.empty_like(y_e3, dtype=float)  # allocate fit.
for level_e3, start_e3, end_e3 in zip(levels_e3, starts_e3, ends_e3):  # expand blocks.
    fit_e3[start_e3:end_e3 + 1] = level_e3  # assign block mean.
print("fit:", np.round(fit_e3, 3))  # inspect fitted staircase.
assert np.all(np.diff(fit_e3) >= -1e-12)  # final monotone check.
plt.figure(figsize=(4, 3))  # fit plot.
plt.plot(y_e3, "o--", label="raw")  # raw values.
plt.step(np.arange(len(fit_e3)), fit_e3, where="mid", color="seagreen", label="isotonic")  # PAVA fit.
plt.title("Easy 3: PAVA fit")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the fitted line is a nondecreasing staircase.

👀 Takeaway: PAVA gives the least-squares monotone projection through repeated pooling.

### Easy 4 — Compare raw and isotonic errors

**Goal.** Compare an unconstrained noisy sequence with its monotone projection, because shape constraints trade flexibility for stability. We build it in 3 steps.

In [ ]:
target_e4 = np.linspace(1, 5, 8)  # smooth monotone target.
raw_e4 = np.array([1.0, 2.2, 1.7, 3.0, 2.6, 4.2, 4.0, 5.1])  # noisy estimates.
iso_e4 = fit_e3.copy()  # isotonic projection from Easy 3.
print("raw monotone?", bool(np.all(np.diff(raw_e4) >= 0)))  # check raw order.
print("iso monotone?", bool(np.all(np.diff(iso_e4) >= 0)))  # check fit order.

▶ What you'll see: only the isotonic sequence respects monotonicity.

In [ ]:
rmse_raw_e4 = float(np.sqrt(np.mean((raw_e4 - target_e4) ** 2)))  # raw RMSE.
rmse_iso_e4 = float(np.sqrt(np.mean((iso_e4 - target_e4) ** 2)))  # isotonic RMSE.
print("raw RMSE:", round(rmse_raw_e4, 3), "iso RMSE:", round(rmse_iso_e4, 3))  # compare errors.
assert rmse_iso_e4 <= rmse_raw_e4 + 0.2  # stability sanity check.

In [ ]:
plt.figure(figsize=(4, 3))  # comparison plot.
plt.plot(target_e4, color="black", label="smooth target")  # target.
plt.plot(raw_e4, "o--", color="gray", label="raw")  # raw estimates.
plt.step(np.arange(len(iso_e4)), iso_e4, where="mid", color="seagreen", label="isotonic")  # constrained fit.
plt.title("Easy 4: monotone stabilization")  # title.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: isotonic regression removes downward wiggles while staying near the trend.

👀 Takeaway: constraints can improve future behavior when they match the problem structure.

### Easy 5 — Reproduce the final score comparison

**Goal.** Compute raw risk, cost, gap, relative gap, and stabilized score, because the full score is the unit of model choice. We build it in 3 steps.

In [ ]:
losses_e5 = np.array([0.213, 0.083, 0.437])  # verified toy losses.
raw_e5 = round(float(losses_e5.mean()), 3)  # empirical risk, rounded like the lesson arithmetic.
cost_e5 = 0.050  # cost term.
score_e5 = raw_e5 + cost_e5  # baseline full score.
print("raw:", round(raw_e5, 3), "score:", round(score_e5, 3))  # inspect numbers.
assert round(raw_e5, 3) == 0.244 and round(score_e5, 3) == 0.294  # checks.

▶ What you'll see: the baseline score is 0.294 after adding cost.

In [ ]:
flex_e5 = 0.338  # flexible alternative score.
gap_e5 = flex_e5 - score_e5  # absolute gap.
rel_e5 = gap_e5 / flex_e5  # relative gap.
stable_e5 = 0.80 * score_e5  # stabilized score.
print("gap:", round(gap_e5, 3), "relative:", round(rel_e5, 3), "stable:", round(stable_e5, 3))  # inspect comparison.
assert round(gap_e5, 3) == 0.044 and round(rel_e5, 3) == 0.130 and round(stable_e5, 3) == 0.235  # lesson checks.

In [ ]:
labels_e5 = ["baseline", "flexible", "stable"]  # options.
scores_e5 = np.array([score_e5, flex_e5, stable_e5])  # full scores.
winner_e5 = labels_e5[int(np.argmin(scores_e5))]  # lower is better.
print("winner:", winner_e5)  # inspect final decision.
plt.figure(figsize=(4, 3))  # comparison chart.
plt.bar(labels_e5, scores_e5, color=["steelblue", "orange", "seagreen"])  # full score bars.
plt.title("Easy 5: decision scores")  # title.
plt.ylabel("score")  # y-axis.
plt.show()  # display.

▶ What you'll see: the stabilized option wins in this toy arithmetic.

👀 Takeaway: compare the score implied by the method, not an isolated training fragment.

## 🔴 Advanced

### Advanced 1 — Build a quantile prediction band

**Goal.** Fit lower and upper quantile lines, because two percentiles form an uncertainty band around the conditional response. We build it in 4 steps.

In [ ]:
x_a1 = np.linspace(0, 5, 11)  # feature grid.
y_a1 = 1.0 + 0.8 * x_a1 + np.array([-0.6, -0.2, 0.1, -0.3, 0.4, 0.2, 0.8, 0.5, 1.1, 0.9, 1.5])  # increasing data with spread.
taus_a1 = [0.2, 0.8]  # lower and upper band edges.
print("points:", len(x_a1), "taus:", taus_a1)  # inspect setup.

▶ What you'll see: the same data will be fit at two quantile levels.

In [ ]:
coefs_a1 = []  # fitted line coefficients.
for tau_a1 in taus_a1:  # fit each edge.
    best_loss_a1 = np.inf  # best risk.
    best_coef_a1 = None  # best coefficients.
    for b1_a1 in np.linspace(0.5, 1.2, 71):  # slopes.
        for b0_a1 in np.linspace(0.2, 1.8, 81):  # intercepts.
            pred_a1 = b0_a1 + b1_a1 * x_a1  # line.
            r_a1 = y_a1 - pred_a1  # residuals.
            loss_a1 = np.where(r_a1 >= 0, tau_a1 * r_a1, (tau_a1 - 1) * r_a1).mean()  # pinball risk.
            if loss_a1 < best_loss_a1:  # update best.
                best_loss_a1 = float(loss_a1)  # save risk.
                best_coef_a1 = np.array([b0_a1, b1_a1])  # save coefficients.
    coefs_a1.append(best_coef_a1)  # store edge.
coefs_a1 = np.array(coefs_a1)  # array form.
print("coefficients:\n", np.round(coefs_a1, 3))  # inspect fits.

In [ ]:
lower_a1 = coefs_a1[0, 0] + coefs_a1[0, 1] * x_a1  # lower predictions.
upper_a1 = coefs_a1[1, 0] + coefs_a1[1, 1] * x_a1  # upper predictions.
coverage_a1 = float(np.mean((y_a1 >= lower_a1) & (y_a1 <= upper_a1)))  # band coverage.
print("band coverage:", round(coverage_a1, 3))  # inspect interval behavior.
assert np.all(upper_a1 >= lower_a1 - 1e-9)  # no crossing on this grid.

In [ ]:
plt.figure(figsize=(5, 3))  # band plot.
plt.scatter(x_a1, y_a1, color="black", label="data")  # observations.
plt.plot(x_a1, lower_a1, color="steelblue", label="tau=.2")  # lower edge.
plt.plot(x_a1, upper_a1, color="crimson", label="tau=.8")  # upper edge.
plt.fill_between(x_a1, lower_a1, upper_a1, color="gray", alpha=0.2)  # interval shade.
plt.title("Advanced 1: quantile band")  # title.
plt.xlabel("x")  # x-axis.
plt.ylabel("y")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: a shaded band between the fitted lower and upper conditional quantiles.

👀 Takeaway: quantile regression can model conditional uncertainty without assuming normal errors.

### Advanced 2 — Choose tau with validation loss

**Goal.** Compare candidate quantile levels on held-out pinball loss, because the desired percentile is a decision choice. We build it in 4 steps.

In [ ]:
x_a2 = np.array([0., 1., 2., 3., 4., 5., 6., 7.])  # feature.
y_a2 = np.array([1.0, 1.7, 2.5, 3.1, 5.6, 5.0, 7.8, 8.2])  # response.
train_a2 = np.array([0, 1, 2, 3, 4, 5])  # training rows.
valid_a2 = np.array([6, 7])  # validation rows.
taus_a2 = np.array([0.2, 0.5, 0.8])  # candidate percentiles.
print("validation targets:", y_a2[valid_a2])  # inspect future points.

▶ What you'll see: two held-out points drive the selection.

In [ ]:
valid_losses_a2 = []  # validation score per tau.
for tau_a2 in taus_a2:  # train one model per tau.
    best_loss_a2 = np.inf  # best training risk.
    best_coef_a2 = None  # best line.
    for b1_a2 in np.linspace(0.7, 1.4, 36):  # slopes.
        for b0_a2 in np.linspace(0.4, 1.8, 29):  # intercepts.
            pred_train_a2 = b0_a2 + b1_a2 * x_a2[train_a2]  # train predictions.
            r_train_a2 = y_a2[train_a2] - pred_train_a2  # train residuals.
            train_loss_a2 = np.where(r_train_a2 >= 0, tau_a2 * r_train_a2, (tau_a2 - 1) * r_train_a2).mean()  # train risk.
            if train_loss_a2 < best_loss_a2:  # update best.
                best_loss_a2 = float(train_loss_a2)  # save risk.
                best_coef_a2 = np.array([b0_a2, b1_a2])  # save line.
    pred_valid_a2 = best_coef_a2[0] + best_coef_a2[1] * x_a2[valid_a2]  # validation predictions.
    r_valid_a2 = y_a2[valid_a2] - pred_valid_a2  # validation residuals.
    valid_losses_a2.append(float(np.where(r_valid_a2 >= 0, tau_a2 * r_valid_a2, (tau_a2 - 1) * r_valid_a2).mean()))  # validation risk.
print("validation losses:", np.round(valid_losses_a2, 3))  # inspect scores.

In [ ]:
best_idx_a2 = int(np.argmin(valid_losses_a2))  # lowest validation risk.
best_tau_a2 = float(taus_a2[best_idx_a2])  # selected tau.
print("selected tau:", best_tau_a2)  # inspect choice.
assert best_tau_a2 in [0.2, 0.5, 0.8]  # candidate check.

In [ ]:
plt.figure(figsize=(5, 3))  # validation sweep plot.
plt.plot(taus_a2, valid_losses_a2, marker="o", color="purple")  # losses.
plt.axvline(best_tau_a2, color="red", linestyle="--", label="selected")  # chosen tau.
plt.title("Advanced 2: validation selects tau")  # title.
plt.xlabel("tau")  # x-axis.
plt.ylabel("held-out pinball loss")  # y-axis.
plt.legend()  # label selected line.
plt.show()  # display.

▶ What you'll see: one quantile level gives the smallest held-out loss.

👀 Takeaway: `tau` should reflect downstream costs and can be selected with validation data.

### Advanced 3 — Run weighted isotonic regression

**Goal.** Add weights to PAVA, because some binned estimates are backed by more data and should pull pooled averages harder. We build it in 4 steps.

In [ ]:
y_a3 = np.array([1.0, 2.4, 1.8, 3.2, 3.0, 4.5])  # bin estimates.
w_a3 = np.array([5., 2., 8., 3., 6., 4.])  # bin sample sizes.
levels_a3 = list(y_a3.astype(float))  # block weighted means.
weights_a3 = list(w_a3.astype(float))  # block weights.
starts_a3 = list(range(len(y_a3)))  # starts.
ends_a3 = list(range(len(y_a3)))  # ends.
print("values:", y_a3, "weights:", w_a3)  # inspect inputs.

▶ What you'll see: each value has a different evidence weight.

In [ ]:
i_a3 = 0  # scan pointer.
while i_a3 < len(levels_a3) - 1:  # scan adjacent blocks.
    if levels_a3[i_a3] > levels_a3[i_a3 + 1]:  # violation.
        total_a3 = weights_a3[i_a3] + weights_a3[i_a3 + 1]  # combined weight.
        avg_a3 = (levels_a3[i_a3] * weights_a3[i_a3] + levels_a3[i_a3 + 1] * weights_a3[i_a3 + 1]) / total_a3  # weighted average.
        levels_a3[i_a3:i_a3 + 2] = [avg_a3]  # merge levels.
        weights_a3[i_a3:i_a3 + 2] = [total_a3]  # merge weights.
        ends_a3[i_a3:i_a3 + 2] = [ends_a3[i_a3 + 1]]  # merge end.
        starts_a3[i_a3:i_a3 + 2] = [starts_a3[i_a3]]  # merge start.
        i_a3 = max(i_a3 - 1, 0)  # recheck left.
    else:
        i_a3 += 1  # move right.
print("weighted levels:", np.round(levels_a3, 3))  # inspect pooled levels.
assert np.all(np.diff(levels_a3) >= -1e-12)  # monotone blocks.

In [ ]:
fit_a3 = np.empty_like(y_a3, dtype=float)  # allocate fit.
for level_a3, start_a3, end_a3 in zip(levels_a3, starts_a3, ends_a3):  # expand blocks.
    fit_a3[start_a3:end_a3 + 1] = level_a3  # fill block.
weighted_sse_a3 = float(np.sum(w_a3 * (y_a3 - fit_a3) ** 2))  # weighted squared error.
print("fit:", np.round(fit_a3, 3), "weighted SSE:", round(weighted_sse_a3, 3))  # inspect result.
assert np.all(np.diff(fit_a3) >= -1e-12)  # monotone fit.

In [ ]:
plt.figure(figsize=(5, 3))  # weighted fit plot.
plt.scatter(np.arange(len(y_a3)), y_a3, s=20 * w_a3, color="gray", label="weighted bins")  # bubble size = weight.
plt.step(np.arange(len(fit_a3)), fit_a3, where="mid", color="seagreen", label="weighted isotonic")  # fit.
plt.title("Advanced 3: weighted PAVA")  # title.
plt.xlabel("sorted bin")  # x-axis.
plt.ylabel("estimate")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: larger bubbles pull the pooled level more strongly.

👀 Takeaway: weighted isotonic regression is the same pooling logic with evidence strength in the average.

### Advanced 4 — Calibrate probabilities with isotonic regression

**Goal.** Turn noisy score bins into a monotone calibration curve, because higher model scores should not imply lower observed probabilities. We build it in 4 steps.

In [ ]:
scores_a4 = np.array([0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75])  # sorted scores.
rates_a4 = np.array([0.08, 0.20, 0.18, 0.42, 0.36, 0.60, 0.58, 0.78])  # observed rates.
print("raw rates:", rates_a4)  # inspect calibration bins.
print("violations:", np.where(np.diff(rates_a4) < 0)[0])  # downward jumps.

▶ What you'll see: some observed rates decrease as score increases.

In [ ]:
levels_a4 = list(rates_a4.astype(float))  # block rates.
weights_a4 = [1.0] * len(rates_a4)  # equal weights.
starts_a4 = list(range(len(rates_a4)))  # starts.
ends_a4 = list(range(len(rates_a4)))  # ends.
i_a4 = 0  # scan pointer.
while i_a4 < len(levels_a4) - 1:  # PAVA scan.
    if levels_a4[i_a4] > levels_a4[i_a4 + 1]:  # violation.
        total_a4 = weights_a4[i_a4] + weights_a4[i_a4 + 1]  # combined weight.
        avg_a4 = (levels_a4[i_a4] * weights_a4[i_a4] + levels_a4[i_a4 + 1] * weights_a4[i_a4 + 1]) / total_a4  # pooled rate.
        levels_a4[i_a4:i_a4 + 2] = [avg_a4]  # merge levels.
        weights_a4[i_a4:i_a4 + 2] = [total_a4]  # merge weights.
        ends_a4[i_a4:i_a4 + 2] = [ends_a4[i_a4 + 1]]  # merge end.
        starts_a4[i_a4:i_a4 + 2] = [starts_a4[i_a4]]  # merge start.
        i_a4 = max(i_a4 - 1, 0)  # recheck left.
    else:
        i_a4 += 1  # move right.
print("calibration levels:", np.round(levels_a4, 3))  # inspect blocks.

In [ ]:
cal_a4 = np.empty_like(rates_a4, dtype=float)  # allocate calibrated rates.
for level_a4, start_a4, end_a4 in zip(levels_a4, starts_a4, ends_a4):  # expand blocks.
    cal_a4[start_a4:end_a4 + 1] = level_a4  # fill calibrated rate.
print("calibrated rates:", np.round(cal_a4, 3))  # inspect final rates.
assert np.all(np.diff(cal_a4) >= -1e-12)  # calibration is monotone.

In [ ]:
plt.figure(figsize=(5, 3))  # calibration plot.
plt.plot(scores_a4, rates_a4, "o--", color="gray", label="raw rate")  # raw bins.
plt.step(scores_a4, cal_a4, where="mid", color="seagreen", label="isotonic")  # monotone calibration.
plt.title("Advanced 4: isotonic calibration")  # title.
plt.xlabel("model score")  # x-axis.
plt.ylabel("observed probability")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: the calibrated curve flattens score regions whose empirical rates were out of order.

👀 Takeaway: isotonic regression is a nonparametric way to calibrate probabilities while preserving score order.

### Advanced 5 — Repair crossing quantiles with isotonic projection

**Goal.** Fix quantile crossing across levels, because an 80th percentile prediction should not be below a 20th percentile prediction for the same example. We build it in 4 steps.

In [ ]:
taus_a5 = np.array([0.2, 0.5, 0.8])  # ordered quantile levels.
preds_a5 = np.array([[2.0, 2.7, 3.1], [3.0, 2.8, 3.6], [4.2, 4.0, 3.9]])  # rows are examples, columns are quantiles.
print("raw predictions:\n", preds_a5)  # inspect predictions.
print("row ordered flags:", [bool(np.all(np.diff(row_a5) >= 0)) for row_a5 in preds_a5])  # crossing check.

▶ What you'll see: some rows are not ordered across quantile levels.

In [ ]:
repaired_a5 = np.empty_like(preds_a5)  # allocate repaired predictions.
for row_idx_a5, row_a5 in enumerate(preds_a5):  # repair each example independently.
    levels_a5 = list(row_a5.astype(float))  # block predictions across tau.
    weights_a5 = [1.0] * len(levels_a5)  # equal confidence.
    starts_a5 = list(range(len(levels_a5)))  # starts.
    ends_a5 = list(range(len(levels_a5)))  # ends.
    i_a5 = 0  # scan pointer.
    while i_a5 < len(levels_a5) - 1:  # PAVA across quantile order.
        if levels_a5[i_a5] > levels_a5[i_a5 + 1]:  # crossing.
            total_a5 = weights_a5[i_a5] + weights_a5[i_a5 + 1]  # combined weight.
            avg_a5 = (levels_a5[i_a5] * weights_a5[i_a5] + levels_a5[i_a5 + 1] * weights_a5[i_a5 + 1]) / total_a5  # pooled value.
            levels_a5[i_a5:i_a5 + 2] = [avg_a5]  # merge levels.
            weights_a5[i_a5:i_a5 + 2] = [total_a5]  # merge weights.
            ends_a5[i_a5:i_a5 + 2] = [ends_a5[i_a5 + 1]]  # merge end.
            starts_a5[i_a5:i_a5 + 2] = [starts_a5[i_a5]]  # merge start.
            i_a5 = max(i_a5 - 1, 0)  # recheck left.
        else:
            i_a5 += 1  # move right.
    for level_a5, start_a5, end_a5 in zip(levels_a5, starts_a5, ends_a5):  # expand repaired row.
        repaired_a5[row_idx_a5, start_a5:end_a5 + 1] = level_a5  # fill block.
print("repaired predictions:\n", np.round(repaired_a5, 3))  # inspect repair.

In [ ]:
ordered_a5 = np.array([np.all(np.diff(row_a5) >= -1e-12) for row_a5 in repaired_a5])  # verify rows.
print("all rows ordered:", bool(np.all(ordered_a5)))  # inspect success.
assert np.all(ordered_a5)  # no quantile crossing remains.

In [ ]:
plt.figure(figsize=(5, 3))  # repair plot.
for idx_a5 in range(preds_a5.shape[0]):  # each example.
    plt.plot(taus_a5, preds_a5[idx_a5], "o--", alpha=0.45, label="raw" if idx_a5 == 0 else None)  # raw quantiles.
    plt.plot(taus_a5, repaired_a5[idx_a5], "s-", label="repaired" if idx_a5 == 0 else None)  # repaired quantiles.
plt.title("Advanced 5: quantile crossing repair")  # title.
plt.xlabel("tau")  # x-axis.
plt.ylabel("prediction")  # y-axis.
plt.legend()  # labels.
plt.show()  # display.

▶ What you'll see: crossing quantile rows are flattened or raised/lowered just enough to become ordered.

👀 Takeaway: isotonic projection is a simple post-processing guardrail for ordered quantile outputs.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Quantile regression changes the target from the mean to a chosen conditional percentile.

Quantile regression changes what the model means: it estimates a conditional percentile instead of the conditional mean. Isotonic regression then enforces a monotone calibration shape, which is useful when the ordering is trustworthy but the raw scores need recalibration.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)
random.seed(7)

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import QuantileRegressor

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs

def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))

def linear_baseline(x_tr, y_tr, x_te):
    clf = LinearRegression()
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)

## The concept, built once on D1

The lesson formula is

$$\rho_\tau(r)=r(\tau-\mathbf 1[r\lt 0])$$

First we reproduce the lesson arithmetic exactly: the three cited losses are 0.213, 0.083, and 0.437.

In [ ]:
lesson_losses = np.array([0.213, 0.083, 0.437])
lesson_total = round(float(lesson_losses.sum()), 3)
lesson_risk = round(lesson_total / 3.0, 3)
lesson_cost = 0.05
lesson_score = round(lesson_risk + lesson_cost, 3)
lesson_alternative = 0.338
lesson_gap = round(lesson_alternative - lesson_score, 3)
lesson_stabilized = round(0.8 * lesson_score, 3)

assert lesson_total == 0.733
assert lesson_risk == 0.244
assert lesson_score == 0.294
assert lesson_gap == 0.044
assert lesson_stabilized == 0.235

print("loss total", lesson_total)
print("empirical risk", lesson_risk)
print("cost-adjusted score", lesson_score)
print("validation gap", lesson_gap)
print("stabilized score", lesson_stabilized)

Now we package the real estimator as `quantile_isotonic_regression_method()`. The method is reusable: it accepts training data and returns predictions for new rows, so the exact same call can run from D1 through D5.

In [ ]:
def quantile_isotonic_regression_method(quantile=0.5, alpha=0.01):
    base = make_pipeline(
        StandardScaler(),
        QuantileRegressor(quantile=quantile, alpha=alpha, solver="highs"),
    )

    def build_and_predict(x_tr, y_tr, x_te):
        base.fit(x_tr, y_tr)
        train_scores = base.predict(x_tr)
        test_scores = base.predict(x_te)
        calibrator = IsotonicRegression(out_of_bounds="clip")
        calibrator.fit(train_scores, y_tr)
        return calibrator.predict(test_scores)

    return build_and_predict

## The dataset ladder

All six notebooks in this batch use `reg_ladder()`: D1 is inspectable, D2 is clean linear signal, D3 is nonlinear sine signal, D4 is sklearn's real diabetes regression dataset, and D5 is a real high-dimensional diabetes interaction design built without downloads. We report MSE as the lesson-plan metric, with `reg_rmse()` as the shared helper check and R² as a secondary annotation.

In [ ]:
rungs = reg_ladder()
diabetes = load_diabetes()
interaction_builder = PolynomialFeatures(degree=2, include_bias=False)
real_d5_X = interaction_builder.fit_transform(diabetes.data)
real_d5_y = diabetes.target
rungs[-1] = ("D5 Diabetes interactions (real, 65-D)", real_d5_X, real_d5_y)

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    print(f"{rung_index}. {name}")
    print("  X shape", X.shape)
    print("  y size", y.shape[0])
    print("  X sample", np.round(X[:3], 3).tolist())
    print("  y sample", np.round(y[:3], 3).tolist())

## Run the same method across D1–D5

We compare median quantile plus isotonic calibration vs mean least squares. The no-skill baseline is `linear_baseline`; the lesson method is `quantile_isotonic_regression_method`. `reg_rmse()` is called on every rung to keep this regression batch tied to the shared helper.

In [ ]:
method = quantile_isotonic_regression_method(quantile=0.5, alpha=0.01)
baseline = linear_baseline
rows = []
prediction_store = []

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    baseline_pred = baseline(x_tr, y_tr, x_te)
    method_pred = method(x_tr, y_tr, x_te)
    baseline_mse = float(mean_squared_error(y_te, baseline_pred))
    method_mse = float(mean_squared_error(y_te, method_pred))
    baseline_rmse = reg_rmse(baseline, X, y)
    method_rmse = reg_rmse(method, X, y)
    assert abs(method_rmse - math.sqrt(method_mse)) < 1e-8
    method_r2 = float(r2_score(y_te, method_pred))
    rows.append((rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2))
    prediction_store.append((name, X, y, x_te, y_te, method_pred))

print("rung | dataset | linear RMSE | method RMSE | method MSE | method R^2")
for rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2 in rows:
    print(f"D{rung_index} | {name} | {baseline_rmse:.3f} | {method_rmse:.3f} | {method_mse:.3f} | {method_r2:.3f}")

## Results visualization

The closing figure has two parts: small multiples show the fitted output artifact on each rung, and the summary curve tracks MSE from D1 to D5.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for col, (name, X, y, x_te, y_te, method_pred) in enumerate(prediction_store):
    axis = axes[0, col]
    order = np.argsort(x_te[:, 0])
    axis.scatter(x_te[:, 0], y_te, s=18, alpha=0.65, label="truth")
    axis.scatter(x_te[:, 0], method_pred, s=18, alpha=0.65, label="prediction")
    axis.set_title(f"D{col + 1} quantile-isotonic calibrated fit", fontsize=9)
    axis.set_xlabel("first feature")
    if col == 0:
        axis.set_ylabel("target")
        axis.legend(fontsize=8)

mse_values = [row[4] for row in rows]
axes[1, 0].plot(range(1, 6), mse_values, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("MSE")
axes[1, 0].set_title("MSE vs ladder complexity")

for empty_axis in axes[1, 1:]:
    empty_axis.axis("off")

plt.tight_layout()
plt.show()

## Pitfall on D5: optimizing the raw term and forgetting the cost

The lesson warns that raw empirical risk is not the whole selection score. On D5 we reproduce that mistake, then add the lesson cost/scale/gap check before choosing the winner.

In [ ]:
d5_name, d5_X, d5_y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(d5_X, d5_y, test_size=0.4, random_state=0)
method_pred = method(x_tr, y_tr, x_te)
baseline_pred = baseline(x_tr, y_tr, x_te)
method_raw = math.sqrt(float(mean_squared_error(y_te, method_pred)))
baseline_raw = math.sqrt(float(mean_squared_error(y_te, baseline_pred)))
wrong_winner = "method" if method_raw < baseline_raw else "linear baseline"
scale = max(method_raw, baseline_raw, 1.0)
method_score = method_raw / scale + lesson_cost
baseline_score = baseline_raw / scale
fixed_winner = "method" if method_score < baseline_score else "linear baseline"
observed_gap = abs(method_score - baseline_score)

print("D5", d5_name)
print("wrong raw-only winner", wrong_winner)
print("method raw RMSE", round(method_raw, 3))
print("baseline raw RMSE", round(baseline_raw, 3))
print("lesson cost", lesson_cost)
print("cost-adjusted method score", round(method_score, 3))
print("cost-adjusted baseline score", round(baseline_score, 3))
print("fixed winner", fixed_winner)
print("observed adjusted gap", round(observed_gap, 3))
print("lesson minimum meaningful gap", lesson_gap)

if observed_gap < lesson_gap:
    print("decision: gap is too small; prefer the simpler setting or collect more validation data")
else:
    print("decision: adjusted gap clears the lesson check")

## Evaluate it + Practice

- Metric: MSE is the lesson-plan metric; `reg_rmse()` supplies the shared RMSE helper check and R² is secondary.
- No-skill baseline: compare against unregularized `linear_baseline` on every rung.
- Cheap sanity check: D1 should be explainable from the printed predictions; if it behaves oddly, inspect preprocessing, extrapolation, and the intercept.
- Ablation: turn off the key idea (median quantile plus isotonic calibration vs mean least squares) and confirm the score or stability worsens on at least one harder rung.
- Failure signals: a tiny validation gap, scale-mismatched scores, or a D5 winner that changes after adding the lesson cost.

Practice 1: change one hyperparameter and rerun the ladder table. Which rung moves the most?

Practice 2: replace RMSE with MAE for the table. Does the D5 winner change?

Practice 3: add a short note explaining whether the lesson gap is large enough for deployment.